# 04 — MDC case study (Q3): naive vs protocol
Plain inline matplotlib. No styled viz (F6/F7 are Day-9 work).

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', 'src'))
from ramanuq.metrics import load_calibrations
from ramanuq.mdc import mdc, to_delta_nd, estimate_sigma_single, estimate_bias

RESULTS = os.path.join('..', 'data', 'synthetic', 'results', 'tierB_grid_results.parquet')
CAL_PATH = os.path.join('..', 'data', 'calibrations', 'calibrations.yaml')
TIERB_DIR = os.path.join('..', 'data', 'synthetic', 'tierB')

df = pd.read_parquet(RESULTS)
cals = load_calibrations(CAL_PATH)

# Excitation wavelength is a fixed property of the suite; read it from truth.
_wls = set()
for _f in os.listdir(TIERB_DIR):
    if _f.endswith('_truth.json'):
        with open(os.path.join(TIERB_DIR, _f)) as _fh:
            _wls.add(json.load(_fh)['wavelength_nm'])
assert len(_wls) == 1, _wls
WAVELENGTH_NM = float(next(iter(_wls)))
MATERIAL_CLASS = 'synthetic_disordered_carbon'
SNR_REGIMES = [15, 50, 200]
CONFIG_COLUMNS = ['baseline', 'lineshape', 'bwf_g', 'peak_set', 'intensity']
WAVELENGTH_NM

In [ ]:
# NAIVE pipeline: linear baseline, single Lorentzian (no BWF), DG peaks, height ratio.
NAIVE_CONFIG = {
    'baseline': 'linear',
    'lineshape': 'lorentzian',
    'bwf_g': False,
    'peak_set': 'DG',
    'intensity': 'height',
}


def select_protocol_config(study_df, snr):
    """PROTOCOL config = DG/area config with the smallest signed-error sd in regime."""
    sub = study_df[
        (study_df.material_class == MATERIAL_CLASS)
        & (study_df.snr_label == snr)
        & (study_df.peak_set == 'DG')
        & (study_df.intensity == 'area')
    ]
    candidates = []
    for keys, grp in sub.groupby(CONFIG_COLUMNS):
        err = grp['error'].to_numpy(dtype=float)
        err = err[np.isfinite(err)]
        if err.size < 2:
            continue
        candidates.append((float(np.std(err, ddof=1)), dict(zip(CONFIG_COLUMNS, keys))))
    candidates.sort(key=lambda t: t[0])
    best_sd, best_cfg = candidates[0]
    return best_cfg, best_sd, candidates

In [ ]:
def _config_str(cfg):
    return '|'.join(f"{k}={cfg[k]}" for k in CONFIG_COLUMNS)


rows = []
for snr in SNR_REGIMES:
    regime = {'material_class': MATERIAL_CLASS, 'snr_label': snr}
    proto_cfg, proto_sd, candidates = select_protocol_config(df, snr)
    print(f'SNR {snr}: selected PROTOCOL = {_config_str(proto_cfg)} '
          f'(smallest DG/area error sd = {proto_sd:.5f} of {len(candidates)} candidates)')

    n_sigma = estimate_sigma_single(df, NAIVE_CONFIG, regime)
    n_bias = estimate_bias(df, NAIVE_CONFIG, regime)
    n_mdc = mdc(n_sigma)
    n_dnd = to_delta_nd(n_mdc, cals, WAVELENGTH_NM)

    p_sigma = estimate_sigma_single(df, proto_cfg, regime)
    p_bias = estimate_bias(df, proto_cfg, regime)
    p_mdc = mdc(p_sigma)
    p_dnd = to_delta_nd(p_mdc, cals, WAVELENGTH_NM)

    rows.append({
        'regime': f'SNR{snr}',
        'naive_config': _config_str(NAIVE_CONFIG),
        'naive_sigma': n_sigma,
        'naive_bias': n_bias,
        'naive_mdc_idig': n_mdc,
        'naive_delta_nd_central': n_dnd[0],
        'protocol_config': _config_str(proto_cfg),
        'protocol_sigma': p_sigma,
        'protocol_bias': p_bias,
        'protocol_mdc_idig': p_mdc,
        'protocol_delta_nd_central': p_dnd[0],
    })

summary = pd.DataFrame(rows)
summary

In [ ]:
cols = [
    'regime',
    'naive_config', 'naive_sigma', 'naive_mdc_idig', 'naive_delta_nd_central',
    'protocol_config', 'protocol_sigma', 'protocol_mdc_idig', 'protocol_delta_nd_central',
]
print(summary[cols].to_string(index=False))

In [ ]:
x = SNR_REGIMES

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, summary['naive_mdc_idig'], marker='o', label='naive')
ax.plot(x, summary['protocol_mdc_idig'], marker='s', label='protocol')
ax.set_xscale('log')
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in x])
ax.set_xlabel('SNR')
ax.set_ylabel('MDC (I_D/I_G units)')
ax.set_title('MDC vs SNR — I_D/I_G currency')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, summary['naive_delta_nd_central'], marker='o', label='naive')
ax.plot(x, summary['protocol_delta_nd_central'], marker='s', label='protocol')
ax.set_xscale('log')
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in x])
ax.set_xlabel('SNR')
ax.set_ylabel('MDC (Delta n_D, cm^-2)')
ax.set_title('MDC vs SNR — Delta n_D currency')
ax.legend()
plt.tight_layout()
plt.show()